# Notebook 04 — Popularity-Based Candidate Retrieval

**Goal:** Build a reproducible, time-safe Top-K popularity baseline for warm users. Compare validation and test using their respective pre-cutoff histories. Evaluate Recall@K and HitRate@K on future view labels, including labels that are unretrievable due to prior interactions or cold items. **No impressions are available; these are not CTR metrics.**

**Protocol:** Validation history `< 2015-08-01`, validation labels `2015-08-01` to `< 2015-08-15`. Test history `< 2015-09-01`, test labels `2015-09-01` to `< 2015-09-15`. No fitting on future labels. This notebook implements a *non-personalized* baseline with per-user already-seen filtering.

## 1. Load data and verify files

In [1]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
if (cwd / 'data' / 'processed').exists():
    ROOT = cwd
elif (cwd.parent / 'data' / 'processed').exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError(f'Could not locate project data/processed from {cwd}')
PROCESSED = ROOT / 'data' / 'processed'
REPORTS = ROOT / 'reports'
REPORTS.mkdir(parents=True, exist_ok=True)

names = {
    'val_history':'train_history.csv',
    'val_labels':'validation_view_labels_warm_users.csv',
    'test_history':'test_history.csv',
    'test_labels':'test_view_labels_warm_users.csv',
}
d = {}
for key, filename in names.items():
    path = PROCESSED / filename
    assert path.is_file(), f'Missing {path}'
    d[key] = pd.read_csv(path)
    print(f'{key}: {len(d[key]):,} rows')
print('Project root:', ROOT)

val_history: 1,899,029 rows
val_labels: 28,634 rows
test_history: 2,452,391 rows
test_labels: 31,876 rows
Project root: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks


## 2. Verify temporal boundaries and label eligibility

Each validation/test label is a distinct `(visitorid, itemid)` pair from the corresponding future window, limited to users observed in history. **Warm users** does not imply **warm items**. We preserve all warm-user positives for the primary denominator, including impossible-to-retrieve cold items, to make the limitation visible.

In [2]:
VAL_CUTOFF_MS = int(pd.Timestamp('2015-08-01', tz='UTC').timestamp() * 1000)
TEST_CUTOFF_MS = int(pd.Timestamp('2015-09-01', tz='UTC').timestamp() * 1000)
for split, cutoff in [('val', VAL_CUTOFF_MS), ('test', TEST_CUTOFF_MS)]:
    history = d[f'{split}_history']
    labels = d[f'{split}_labels']
    assert history.timestamp.max() < cutoff, f'{split} history crosses cutoff'
    assert labels[['visitorid','itemid']].notna().all().all()
    assert not labels[['visitorid','itemid']].duplicated().any()
    assert labels.visitorid.isin(history.visitorid).all(), 'Warm-user cohort violated'
    print(f'{split}: cutoff verified; {labels.visitorid.nunique():,} evaluation users; {len(labels):,} positive pairs')

val: cutoff verified; 13,329 evaluation users; 28,634 positive pairs
test: cutoff verified; 15,137 evaluation users; 31,876 positive pairs


## 3. Train-only item popularity

Use the number of historical **view events** per item. Using event counts (rather than unique viewers) is an explicit baseline choice; no add-to-cart/purchase weighting is used yet. We compute a separate popularity ranking for each cutoff.

In [3]:
def fit_popularity(history):
    counts = (history.loc[history.event.eq('view'), 'itemid']
              .value_counts().rename_axis('itemid').reset_index(name='view_count'))
    # Stable tie-breaking for reproducibility
    counts = counts.sort_values(['view_count','itemid'], ascending=[False,True], kind='mergesort')
    return counts.reset_index(drop=True)

val_pop = fit_popularity(d['val_history'])
test_pop = fit_popularity(d['test_history'])
print('Validation top 10:')
display(val_pop.head(10))
print('Test top 10:')
display(test_pop.head(10))

Validation top 10:


,itemid,view_count
0,5411,1877
1,309778,1520
2,370653,1464
3,298009,1336
4,461686,1271
5,335975,1228
6,257040,1204
7,96924,1080
8,369447,1076
9,111530,1034


Test top 10:


,itemid,view_count
0,187946,2694
1,5411,2165
2,461686,2028
3,370653,1735
4,309778,1560
5,298009,1538
6,257040,1526
7,219512,1482
8,96924,1421
9,335975,1349


## 4. Build user histories and Top-K recommendations

Exclude all items previously interacted with by the user (not just prior views). This is a design choice for **novel-item recommendation**. Future labels of already-seen items remain in the primary denominator, so maximum achievable recall can be less than 1. Generate 100 candidates per user; do not create a dense user–item matrix.

In [4]:
MAX_K = 100
K_VALUES = [10, 20, 50, 100]

def make_seen_map(history, users):
    selected = history.loc[history.visitorid.isin(users), ['visitorid','itemid']].drop_duplicates()
    return selected.groupby('visitorid')['itemid'].agg(set).to_dict()

def recommend_popularity(popularity, seen_map, users, k=100):
    ranked = popularity.itemid.to_numpy().tolist()
    recommendations = {}
    for user in users:
        seen = seen_map.get(user, set())
        recs = []
        for item in ranked:
            if item not in seen:
                recs.append(int(item))
                if len(recs) == k:
                    break
        recommendations[int(user)] = recs
    return recommendations

results = {}
for split in ['val', 'test']:
    history, labels = d[f'{split}_history'], d[f'{split}_labels']
    users = sorted(int(u) for u in labels.visitorid.unique())
    seen_map = make_seen_map(history, users)
    t0 = time.perf_counter()
    recs = recommend_popularity(val_pop if split=='val' else test_pop, seen_map, users, MAX_K)
    elapsed = time.perf_counter() - t0
    results[split] = {'users':users, 'seen':seen_map, 'recs':recs, 'elapsed_s':elapsed}
    assert all(len(v)==MAX_K for v in recs.values()), 'Fewer than 100 candidates'
    assert all(not set(v).intersection(seen_map.get(u,set())) for u,v in recs.items())
    print(f'{split}: generated {MAX_K} candidates for {len(users):,} users in {elapsed:.2f}s '
          f'({elapsed / max(1,len(users)) * 1000:.3f} ms/user batch average)')

# This is a batch generation throughput estimate, NOT online API p95 latency.

val: generated 100 candidates for 13,329 users in 0.34s (0.025 ms/user batch average)
test: generated 100 candidates for 15,137 users in 0.35s (0.023 ms/user batch average)


## 5. Offline evaluation: Recall@K and HitRate@K

For each user, `Recall@K = (# future positive items retrieved in Top-K) / (# future positive items)`; `HitRate@K` is 1 if any future positive is in Top-K, else 0. Report macro-average across users, plus micro recall across positive pairs. Because the evaluation uses logged interactions rather than exposed recommendation impressions, these are *offline proxies*, not unbiased estimates of online recommendation performance.

In [5]:
def evaluate(recs, labels, seen_map, history_items, k_values):
    positives = labels.groupby('visitorid')['itemid'].agg(set).to_dict()
    rows = []
    for k in k_values:
        recalls, hits = [], []
        total_found = 0
        total_pos = 0
        for user, truth in positives.items():
            recommended = set(recs[int(user)][:k])
            found = len(recommended & truth)
            recalls.append(found / len(truth))
            hits.append(float(found > 0))
            total_found += found
            total_pos += len(truth)
        rows.append({'K':k, 'Recall@K_macro':np.mean(recalls),
                     'Recall@K_micro':total_found/total_pos,
                     'HitRate@K':np.mean(hits),
                     'UniqueRecommendedItems':len({item for r in recs.values() for item in r[:k]}),
                     'CatalogCoverage':len({item for r in recs.values() for item in r[:k]})/len(history_items)})
    # Eligibility upper bound: cold historical items and previously seen positives cannot be retrieved
    eligible = sum(1 for u, i in labels[['visitorid','itemid']].itertuples(index=False, name=None)
                   if i in history_items and i not in seen_map.get(int(u),set()))
    print(f'Positive pairs: {len(labels):,}; eligible after catalog + seen filters: {eligible:,} '
          f'({eligible/len(labels):.2%} maximum possible micro recall under this policy)')
    return pd.DataFrame(rows), eligible

metrics = {}
for split in ['val','test']:
    history = d[f'{split}_history']
    labels = d[f'{split}_labels']
    m, eligible = evaluate(results[split]['recs'], labels, results[split]['seen'],
                           set(history.itemid.unique()), K_VALUES)
    metrics[split] = m
    results[split]['eligible_pairs'] = eligible
    print(f'\n{split.upper()}')
    display(m.round(6))

Positive pairs: 28,634; eligible after catalog + seen filters: 21,984 (76.78% maximum possible micro recall under this policy)

VAL


,K,Recall@K_macro,Recall@K_micro,HitRate@K,UniqueRecommendedItems,CatalogCoverage
0,10,0.003415,0.002968,0.006077,17,0.000085
1,20,0.006215,0.005727,0.010728,38,0.000190
2,50,0.011794,0.011106,0.019731,86,0.000431
3,100,0.018470,0.017636,0.030235,164,0.000821


Positive pairs: 31,876; eligible after catalog + seen filters: 24,875 (78.04% maximum possible micro recall under this policy)

TEST


,K,Recall@K_macro,Recall@K_micro,HitRate@K,UniqueRecommendedItems,CatalogCoverage
0,10,0.003659,0.002384,0.004955,18,0.000081
1,20,0.006871,0.005176,0.010372,42,0.000188
2,50,0.012874,0.011325,0.021008,97,0.000435
3,100,0.019363,0.017411,0.031314,184,0.000825


## 6. Save metrics and sample recommendations

Only the **validation** results should guide method selection. Test results here are reported as a baseline snapshot; avoid repeated tuning against test.

In [6]:
for split in ['val','test']:
    metrics[split].to_csv(REPORTS / f'notebook04_{split}_popularity_metrics.csv', index=False)
    sample_users = results[split]['users'][:20]
    sample = pd.DataFrame([(u, rank, item) for u in sample_users
                           for rank,item in enumerate(results[split]['recs'][u],start=1)],
                          columns=['visitorid','rank','itemid'])
    sample.to_csv(REPORTS / f'notebook04_{split}_popularity_sample.csv', index=False)

summary = {
    'method':'historical_view_count_popularity',
    'protocol':'warm-user future-view pairs; exclude historically seen items; labels include cold/seen positives',
    'cutoffs_utc':{'validation':'2015-08-01','test':'2015-09-01'},
    'K_values':K_VALUES,
    'validation':{'users':len(results['val']['users']), 'positive_pairs':len(d['val_labels']),
                  'eligible_pairs':results['val']['eligible_pairs'],
                  'batch_seconds':results['val']['elapsed_s'],
                  'metrics':metrics['val'].to_dict(orient='records')},
    'test':{'users':len(results['test']['users']), 'positive_pairs':len(d['test_labels']),
            'eligible_pairs':results['test']['eligible_pairs'],
            'batch_seconds':results['test']['elapsed_s'],
            'metrics':metrics['test'].to_dict(orient='records')},
}
path = REPORTS / 'notebook04_popularity_baseline.json'
path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('Saved:', path)
print('PASS: temporal boundaries, warm users, candidate uniqueness/seen filtering, and metric exports')

Saved: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\reports\notebook04_popularity_baseline.json
PASS: temporal boundaries, warm users, candidate uniqueness/seen filtering, and metric exports


## 7. Interpretation and next steps

- Compare **Recall@50** and **Recall@100** on validation with later Item-CF and embedding retrieval.
- Macro recall weights users equally; micro recall weights positive pairs equally.
- CatalogCoverage is the fraction of historical catalog items that appear at least once in recommendations, and can be low for global popularity ranking.
- The eligible-pair ceiling quantifies cold-item and repeat-item limitations under our chosen policy.
- `batch_seconds / user_count` is **not** p95 online request latency; Notebook 15 will measure serving latency separately.
- **Next:** Notebook 05 — Item-Based Collaborative Filtering Retrieval, using historical user–item co-occurrence and the same evaluation protocol.